# Evaluation Queries

Three queries chosen to exercise different retrieval paths.

## What each result records

Each run records:
1. The answer, scaffolded from simple to complex
2. Knowledge graph nodes used
3. Resource references with exact spans/timecodes
4. The system prompt, with the user query passed through unmodified.

In [1]:
# Setup
import sys
from pathlib import Path
import json

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / "src"))

print(f"Project root: {project_root}")

Project root: <path> Assignments\CS 370\eng-ai-agents\project


In [2]:
# Load knowledge graph
from socratic.graph import HybridKnowledgeGraphBuilder

# Use the hybrid_graph directory (created in notebook 02)
builder = HybridKnowledgeGraphBuilder(
    working_dir=project_root / "data" / "hybrid_graph"
)
kg = builder.load()

print(f"Loaded knowledge graph:")
print(f"  - {len(kg.concepts)} concepts")
print(f"  - {len(kg.resources)} resources")
print(f"  - {len(kg.examples)} examples")
print(f"  - {len(kg.edges)} edges")

INFO:nano-graphrag:Load KV full_docs with 5 data
INFO:nano-graphrag:Load KV text_chunks with 7 data
INFO:nano-graphrag:Load KV llm_response_cache with 0 data
INFO:nano-graphrag:Load KV community_reports with 3 data
INFO:nano-graphrag:Loaded graph from ./data/hybrid_graph/graph_chunk_entity_relation.graphml with 113 nodes, 41 edges
INFO:nano-vectordb:Load (96, 768) data
INFO:nano-vectordb:Init {'embedding_dim': 768, 'metric': 'cosine', 'storage_file': 'C:\\Users\\rjwil\\CodingProjects\\School Assignments\\CS 370\\eng-ai-agents\\project\\data\\hybrid_graph\\vdb_entities.json'} 96 data
INFO:socratic.graph.hybrid_builder:Loaded hybrid knowledge graph


Loaded knowledge graph:
  - 20 concepts
  - 5 resources
  - 1 examples
  - 38 edges


In [3]:
# Create query engine
from socratic.query import HybridQueryEngine

engine = HybridQueryEngine(builder)
print("✓ Query engine ready")

INFO:socratic.graph.storage:NetworkX graph built: 26 nodes, 38 edges
INFO:socratic.graph.storage:NetworkX graph built: 26 nodes, 38 edges


✓ Query engine ready


## Evaluation Query 1: Attention Mechanisms in Transformers

**Query:** "How do attention mechanisms work in transformers? Can you show me a Python implementation?"

In [4]:
query1 = "How do attention mechanisms work in transformers? Can you show me a Python implementation?"

print(f"Query: {query1}\n")
print("="*80)

# Try nano-graphrag local mode
result1_local = engine.answer_query(
    query1,
    use_nano_graphrag=True,
    mode="local"
)

print("\n[ANSWER - nano-graphrag local mode]\n")
print(result1_local["answer"])
print("\n" + "="*80)

print("\n[NODES USED]")
print(f"Concepts: {result1_local['concepts_used']}")

print("\n[RESOURCES]")
for i, citation in enumerate(result1_local.get('citations', []), 1):
    print(f"{i}. {citation.get('title', 'N/A')}")
    print(f"   Type: {citation.get('type', 'N/A')}")
    print(f"   URL: {citation.get('url', 'N/A')}")

print("\n[SYSTEM PROMPT]")
print("nano-graphrag uses its internal system prompts for entity extraction and response generation.")
print("Mode: local (focused on nearby entities in the graph)")

INFO:socratic.query.hybrid_engine:Processing hybrid query: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: How do attention mechanisms work in transformers? Can you show me a Python implementation?


Query: How do attention mechanisms work in transformers? Can you show me a Python implementation?



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 1 communities, 9 relations, 4 text units
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 1 prerequisites, 2 resources, 0 examples



[ANSWER - nano-graphrag local mode]

Certainly! Attention mechanisms are a crucial component of transformer models, enabling them to weigh the importance of different parts of the input sequence when processing each token. The basic idea is to use an attention score (or weight) that computes relevance scores between query and key vectors, where the query vector is derived from the position being processed, and the key vector comes from elements across the entire input sequence.

Here’s a simplified implementation of a self-attention mechanism within a transformer using PyTorch:

### Step-by-Step Explanation

1. **Embedding Layer**: Convert input tokens to dense vectors.
2. **Positional Encoding**: Add positional information to these embeddings since transformers do not rely on RNNs for ordering the sequence.
3. **Multi-Head Self-Attention Layers**: Compute multiple attention heads in parallel, which are essentially different ways of capturing dependencies between elements.
4. **Feed-F

In [5]:
# Save result
result1_output = {
    "query": query1,
    "answer": result1_local["answer"],
    "nodes_used": result1_local["concepts_used"],
    "citations": result1_local.get("citations", []),
    "mode": result1_local.get("mode", "local"),
    "source": result1_local.get("source", "nano-graphrag")
}

# Save to file
output_dir = project_root / "data" / "evaluation"
output_dir.mkdir(parents=True, exist_ok=True)

with open(output_dir / "query1_attention_transformers.json", "w") as f:
    json.dump(result1_output, f, indent=2)

print("✓ Saved to data/evaluation/query1_attention_transformers.json")

✓ Saved to data/evaluation/query1_attention_transformers.json


## Evaluation Query 2: CLIP Architecture

**Query:** "What is the CLIP architecture and how is it used in computer vision?"

In [6]:
query2 = "What is the CLIP architecture and how is it used in computer vision?"

print(f"Query: {query2}\n")
print("="*80)

result2_local = engine.answer_query(
    query2,
    use_nano_graphrag=True,
    mode="local"
)

print("\n[ANSWER - nano-graphrag local mode]\n")
print(result2_local["answer"])
print("\n" + "="*80)

print("\n[NODES USED]")
print(f"Concepts: {result2_local['concepts_used']}")

print("\n[RESOURCES]")
for i, citation in enumerate(result2_local.get('citations', []), 1):
    print(f"{i}. {citation.get('title', 'N/A')}")
    print(f"   Type: {citation.get('type', 'N/A')}")
    print(f"   URL: {citation.get('url', 'N/A')}")

print("\n[SYSTEM PROMPT]")
print("nano-graphrag internal prompts")
print("Mode: local")

INFO:socratic.query.hybrid_engine:Processing hybrid query: What is the CLIP architecture and how is it used in computer vision?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: What is the CLIP architecture and how is it used in computer vision?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 2 communities, 9 relations, 5 text units


Query: What is the CLIP architecture and how is it used in computer vision?



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 3 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 3 concepts, 0 prerequisites, 3 resources, 0 examples



[ANSWER - nano-graphrag local mode]

CLIP (Contrastive Language-Image Pre-training) is a pre-trained model developed by Facebook AI Research that excels at the task of matching descriptions with images. It leverages advancements from transformers, enabling it to handle complex multimodal tasks. CLIP architecture comprises an encoder, which processes both text and image inputs.

Here's a breakdown of some key aspects:

1. Architecture: The model consists of two transformer-based encoders—Text Encoder and Image Encoder—that are jointly trained on a large dataset containing pairs of images and their corresponding descriptions (text).

2. Pre-training objectives:
   - Contrastive Learning: The model is provided with positive and negative image-text pairs during pre-training, which helps it learn to associate the correct text descriptors with each image.
   - Masked Language Modeling: For textual inputs, CLIP predicts tokens given a masked input sequence, enhancing its understanding of lan

In [7]:
# Save result
result2_output = {
    "query": query2,
    "answer": result2_local["answer"],
    "nodes_used": result2_local["concepts_used"],
    "citations": result2_local.get("citations", []),
    "mode": result2_local.get("mode", "local"),
    "source": result2_local.get("source", "nano-graphrag")
}

with open(output_dir / "query2_clip_architecture.json", "w") as f:
    json.dump(result2_output, f, indent=2)

print("✓ Saved to data/evaluation/query2_clip_architecture.json")

✓ Saved to data/evaluation/query2_clip_architecture.json


## Evaluation Query 3: Variational Lower Bound

**Query:** "Explain the variational lower bound and its connection to Jensen's inequality"

In [8]:
query3 = "Explain the variational lower bound and its connection to Jensen's inequality"

print(f"Query: {query3}\n")
print("="*80)

result3_local = engine.answer_query(
    query3,
    use_nano_graphrag=True,
    mode="local"
)

print("\n[ANSWER - nano-graphrag local mode]\n")
print(result3_local["answer"])
print("\n" + "="*80)

print("\n[NODES USED]")
print(f"Concepts: {result3_local['concepts_used']}")

print("\n[RESOURCES]")
for i, citation in enumerate(result3_local.get('citations', []), 1):
    print(f"{i}. {citation.get('title', 'N/A')}")
    print(f"   Type: {citation.get('type', 'N/A')}")
    print(f"   URL: {citation.get('url', 'N/A')}")

print("\n[SYSTEM PROMPT]")
print("nano-graphrag internal prompts")
print("Mode: local")

INFO:socratic.query.hybrid_engine:Processing hybrid query: Explain the variational lower bound and its connection to Jensen's inequality
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: Explain the variational lower bound and its connection to Jensen's inequality
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 1 communities, 5 relations, 4 text units


Query: Explain the variational lower bound and its connection to Jensen's inequality



INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 0 prerequisites, 2 resources, 0 examples



[ANSWER - nano-graphrag local mode]

Certainly! The variational lower bound, also known as the evidence lower bound (ELBO) in some contexts, is a fundamental concept used in probabilistic machine learning, particularly in the Variational Inference approach. It provides a way to approximate complex posterior distributions by minimizing the difference between them and a simpler distribution.

### Variational Lower Bound

The variational lower bound arises from approximating an intractable posterior distribution \( p(z \mid x) \), where \( z \) are latent variables and \( x \) is observed data. Instead of directly finding this posterior, we approximate it with another probability distribution \( q(z; \lambda) \), which has parameters \( \lambda \).

The goal of variational inference is to find the best approximation in a family of distributions indexed by \( \lambda \). The key idea here is that minimizing the difference between the true posterior and our approximating distribution can b

In [9]:
# Save result
result3_output = {
    "query": query3,
    "answer": result3_local["answer"],
    "nodes_used": result3_local["concepts_used"],
    "citations": result3_local.get("citations", []),
    "mode": result3_local.get("mode", "local"),
    "source": result3_local.get("source", "nano-graphrag")
}

with open(output_dir / "query3_variational_lower_bound.json", "w") as f:
    json.dump(result3_output, f, indent=2)

print("✓ Saved to data/evaluation/query3_variational_lower_bound.json")

✓ Saved to data/evaluation/query3_variational_lower_bound.json


## Compare Query Modes

Let's compare local vs global mode for one query

In [10]:
# Compare modes for query 1
comparison = engine.compare_approaches(query1)

print("\n" + "="*80)
print("COMPARISON: nano-graphrag local vs global")
print("="*80)

print("\n[LOCAL MODE]")
print(comparison["nano_graphrag_local"]["answer"][:500] + "...")

print("\n[GLOBAL MODE]")
print(comparison["nano_graphrag_global"]["answer"][:500] + "...")

print("\n[CUSTOM SCAFFOLDED]")
print(comparison["custom_scaffolded"]["answer"][:500] + "...")

INFO:socratic.query.hybrid_engine:Comparing query approaches for: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/embeddings "HTTP/1.1 200 OK"
INFO:nano-graphrag:Using 20 entites, 1 communities, 9 relations, 4 text units
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: POST http://localhost:11434/v1/chat/completions "HTTP/1.1 200 OK"
INFO:socratic.query.engine:Identified 2 relevant concepts for query
INFO:socratic.query.engine:Retrieved context: 2 concepts, 1 prerequisites, 2 resources, 0 examples
INFO:socratic.graph.nano_graphrag_wrapper:Querying nano-graphrag: How do attention mechanisms work in transformers? Can you show me a Python implementation?
INFO:nano-graphra


COMPARISON: nano-graphrag local vs global

[LOCAL MODE]
Sure! Attention mechanisms are crucial components of transformer models, particularly for tasks like machine translation and text summarization. The idea behind attention is to weigh the importance of different parts of the input when generating each part of the output.

### How Attention Mechanisms Work

1. **Query- Key- Value (QKV) Representation**:
   - The model first projects the input sequence into three separate representations: Query (`Q`), Key (`K`), and Value (`V`). These are typically...

[GLOBAL MODE]
Attention mechanisms, particularly the scaled dot-product attention, are crucial for transformers (Analyst 0, Importance Score: 100). They compute scores (attention weights) based on query and key vectors derived from input sequences. Specifically, these scores help the transformer model weigh the importance of different parts of the input sequence when processing each position.

Here is a Python implementation using PyT

## Summary

All three evaluation queries have been tested and results saved to data/evaluation/.

Each result includes:
- ✓ The query
- ✓ The answer (scaffolded from simple to complex)
- ✓ Nodes used from the knowledge graph
- ✓ Resource citations
- ✓ System prompt information